# Train and quantize YOLOv8n-seg for STM32N6

Run the repository's COCO TFLite model on your PC, then fine-tune YOLOv8n-seg on your own images and export a model for the NUCLEO-N657X0-Q. [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb) takes the exported model and its class metadata through deployment.

Use the [segmentation benchmark](segmentation/README.md) to compare FP32 and INT8 TFLite on the same held-out mask test set.

## What you will learn

| Stage | Question it answers | Evidence you should see |
| --- | --- | --- |
| Reference inference | Can my PC load and run a known quantized model? | One input and two output tensors with concrete shapes and types |
| Dataset | Do class IDs and polygons match the images? | Split counts, a dataset YAML, and a polygon overlaid on an image |
| Training | Can the model learn my classes on held-out captures? | A validation-selected PyTorch checkpoint |
| Export and evaluation | Does the integer model preserve the useful predictions? | A TFLite file, a JSON interface description, and test metrics for both models |

Run cells from top to bottom. Sections 1 and 1.1 work with the bundled model before you have a dataset. Read the [architecture walkthrough](../Doc/Application-Overview.md) if terms such as tensor, inference, and NPU are new to you. Training and export run on the PC. The STM32N6 uses the resulting model for inference.

## Python environment

- For the **COCO reference test**, install `ai-edge-litert numpy pillow` in the Jupyter environment. The cell can also use the TensorFlow interpreter if installed.
- For **training and export**, create an environment with Python 3.10 or 3.11. The [ST export tutorial](https://github.com/STMicroelectronics/stm32ai-modelzoo-services/blob/main/instance_segmentation/docs/tuto/How_to_deploy_yolov8_instance_segmentation.md) validated Ultralytics 8.3.88. Start with the versions below. The exporter checks additional conversion dependencies. Save `python -m pip freeze` once the export works.

```bash
python -m pip install "ultralytics==8.3.88" "tensorflow==2.15.0" "onnx==1.15.0" "onnxruntime==1.18.1" "keras==2.15.0" "tf-keras==2.15.0" ipykernel
python -m ipykernel install --user --name stm32n6-yolov8 --display-name "Python 3 (STM32N6 YOLOv8)"
```

Select the new kernel in Jupyter. On Windows, WSL2 is an option if the export stack causes problems. Copy the exported TFLite and class JSON to the machine that runs ST Edge AI Core.

In [ ]:
from pathlib import Path
import json
import numpy as np
from PIL import Image
from IPython.display import display

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "Model").is_dir() else HERE.parent
# REPO = Path(r"C:\projects\STM32N6-Edge_Computer_Vision")
BASELINE = REPO / "Model" / "yolov8n_256_quant_pc_ii_seg_coco-st.tflite"
assert BASELINE.is_file(), f"Set REPO to the repository root. Missing {BASELINE}"

# Keep images and training files outside the repo so code updates leave them alone.
WORK = REPO.parent / "stm32n6_model_work"
DATASET_ROOT = REPO.parent / "insect_dataset"
CLASS_NAMES = ["insect"]  # Match the class IDs in the label files.
# Start with the firmware reference size. Changing this affects the grid, activation memory, and decoder.
IMAGE_SIZE = 256
IMAGE = None  # Optional: Path(r"C:\path\capture.jpg")

## 1. Run the COCO reference model

The bundled TFLite uses signed 8-bit input with a scale of about 1/255 and a zero point of −128. ST Edge AI Core generates an unsigned 8-bit input with zero point 0 for the same model, so the firmware can feed it raw RGB bytes. The conversion is `real = scale × (quantized - zero_point)`.

This test resizes an RGB image directly to 256 × 256, matching the aspect-ratio behavior of the firmware inference pipe. Interpolation and camera ISP can still change individual pixels. To compare inference tensors on the PC and board, use the same input RGB bytes.

Think of the input as a four-axis array. `[1, 256, 256, 3]` means one image, 256 rows, 256 columns, and RGB at each pixel. The interpreter needs integer values in the input tensor's own coordinate system. If a channel is 128, the floating-point model should see about 0.502. With the COCO TFLite input's scale and zero point, this is encoded as signed value 0. The MCU's generated interface represents the same channel with unsigned value 128.

With `IMAGE = None` the notebook passes a uniform image to exercise the loading and invocation code. Use a picture of a COCO object, such as a bottle, to explore the predictions.

In [ ]:
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    from tensorflow.lite import Interpreter

def open_tflite(path):
    interpreter = Interpreter(model_path=str(path), num_threads=2)
    # The interpreter plans and allocates buffers before we write an input or request an output.
    interpreter.allocate_tensors()
    return interpreter

def tensor_info(detail):
    scale, zero_point = detail["quantization"]
    return {
        "name": detail["name"],
        "shape": detail["shape"].tolist(),
        "dtype": np.dtype(detail["dtype"]).name,
        "scale": float(scale),
        "zero_point": int(zero_point),
    }

def run_tflite(interpreter, rgb):
    inputs = interpreter.get_input_details()
    assert len(inputs) == 1, "Expected a single RGB input."
    info = inputs[0]
    scale, zero_point = info["quantization"]
    assert scale > 0 and info["dtype"] in (np.int8, np.uint8)
    limits = np.iinfo(info["dtype"])
    # The leading axis is the batch. Division by 255 recovers the real-valued RGB range used by this graph.
    real = rgb[None].astype(np.float32) / 255
    assert real.shape == tuple(info["shape"])
    # Invert the affine dequantization formula. Round, then clamp before casting to prevent integer wraparound.
    data = np.clip(np.rint(real / scale + zero_point), limits.min, limits.max)
    interpreter.set_tensor(info["index"], data.astype(info["dtype"]))
    interpreter.invoke()
    # Each output has its own shape and affine parameters. Keep the raw values to inspect them below.
    return [interpreter.get_tensor(d["index"]) for d in interpreter.get_output_details()]

baseline = open_tflite(BASELINE)
for detail in [*baseline.get_input_details(), *baseline.get_output_details()]:
    print(tensor_info(detail))

if IMAGE is None:
    rgb = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), 220, dtype=np.uint8)
else:
    reference = Image.open(IMAGE).convert("RGB")
    rgb = np.asarray(reference.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))

raw_outputs = run_tflite(baseline, rgb)
print("COCO outputs:", [(output.shape, output.dtype.name) for output in raw_outputs])

### 1.1. Inspect a candidate before C postprocessing

The detection output stores one row per attribute and one column per candidate. The 116 rows in the COCO model are 4 box values, 80 class scores, and 32 mask coefficients. Read one candidate in Python with the same indexing that the C decoder uses. The highest raw candidate score can come from a background image. NMS and mask reconstruction happen later.

Try a known COCO object and then an empty image. Watch how the best score and center change. The architecture guide explains how the firmware applies thresholds and converts these values into an overlay.

The last lines reconstruct that candidate's mask at prototype resolution. Think of the 32 coefficients as weights for combining 32 shared patterns. The notebook performs the dot product with dequantized floats. The firmware uses integer products and an equivalent scaled threshold up to rounding. The preview shows the mask before confidence filtering, NMS, and clipping to the box.

In [ ]:
detection_details = baseline.get_output_details()[0]
assert tuple(detection_details["shape"]) == (1, 116, 1344)
detection_scale, detection_zero_point = detection_details["quantization"]

# Convert to a wider type before subtraction so int8 values cannot overflow.
detections = (raw_outputs[0][0].astype(np.float32) - detection_zero_point) * detection_scale
class_scores = detections[4:84, :]
class_id, candidate_id = np.unravel_index(class_scores.argmax(), class_scores.shape)
print("Candidate and class ID:", candidate_id, class_id)
print("Class score before thresholding or NMS:", float(class_scores[class_id, candidate_id]))
print("Normalized cx, cy, width, height:", detections[:4, candidate_id].tolist())
coefficients = detections[84:, candidate_id]
print("Mask coefficients for this candidate:", coefficients.shape[0])

prototype_details = baseline.get_output_details()[1]
assert tuple(prototype_details["shape"]) == (1, 64, 64, 32)
prototype_scale, prototype_zero_point = prototype_details["quantization"]
prototypes = (raw_outputs[1][0].astype(np.float32) - prototype_zero_point) * prototype_scale

# Multiply each pixel's 32 prototype values by the same 32 candidate coefficients.
# The dot product collapses the channel axis, yielding one value per mask pixel.
mask_values = prototypes @ coefficients
prototype_mask = (mask_values >= 0.5).astype(np.uint8) * 255
display(Image.fromarray(prototype_mask).resize((256, 256), Image.Resampling.NEAREST))

## 2. Prepare images and polygons

Create `insect_dataset/images/{train,val,test}` and `insect_dataset/labels/{train,val,test}` beside the repository. For each `001.jpg`, add `001.txt` in the corresponding labels folder. Each object has one row with the class ID and the polygon coordinates normalized to the image width and height.

```text
0 0.40 0.44 0.54 0.43 0.55 0.57 0.41 0.55
```

Use an empty TXT file for a background image. Group captures by event, date, or trap before splitting. This keeps nearly identical burst frames from appearing in both train and test. Include empty backgrounds, reflections, motion, and difficult examples. Check whether insects are still visible after resizing to 256 × 256.

[Ultralytics segmentation dataset format](https://docs.ultralytics.com/datasets/segment/)

### Why this structure matters

The training split updates the model weights. The validation split selects the checkpoint and later supplies representative images for calibration in this notebook. The test split estimates performance on images held back from both choices. For a trap camera, split by collection session or trigger event as appropriate. Counting different frame filenames alone does not establish independence.

The dataset check distinguishes a deliberately empty TXT from a missing label. It also checks the polygon row syntax. A syntactically valid polygon can still be on the wrong insect, so use the visualization below and inspect examples from every split.

In [ ]:
import yaml

assert CLASS_NAMES and len(CLASS_NAMES) == len(set(CLASS_NAMES))
assert len(CLASS_NAMES) <= 255, "The demo postprocessor uses 8-bit class IDs."
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
first_test_image = None
training_example = None

for split in ("train", "val", "test"):
    images_root = DATASET_ROOT / "images" / split
    # Mirror subdirectories too, so images with the same stem in different folders keep distinct labels.
    labels_root = DATASET_ROOT / "labels" / split
    images = sorted(p for p in images_root.rglob("*") if p.suffix.lower() in image_extensions)
    assert images, f"No images in {images_root}"
    instances = 0
    backgrounds = 0
    for image_path in images:
        label = labels_root / image_path.relative_to(images_root).with_suffix(".txt")
        assert label.is_file(), f"Missing {label}. Use an empty file for a background image."
        annotation = label.read_text(encoding="utf-8")
        backgrounds += not annotation.strip()
        for row_number, row in enumerate(annotation.splitlines(), 1):
            tokens = row.split()
            if not tokens:
                continue
            location = f"{label} line {row_number}"
            # One class ID followed by at least three complete (x, y) pairs gives at least seven tokens.
            assert len(tokens) >= 7 and len(tokens) % 2 == 1, f"Invalid polygon: {location}"
            assert 0 <= int(tokens[0]) < len(CLASS_NAMES), f"Invalid class ID: {location}"
            assert all(0 <= float(x) <= 1 for x in tokens[1:]), f"Invalid coordinates: {location}"
            if split == "train" and training_example is None:
                training_example = (image_path, label)
            instances += 1
    assert instances, f"The {split} split has no labeled instances."
    print(f"{split}: {len(images)} images, {backgrounds} backgrounds, {instances} instances")
    if split == "test":
        first_test_image = images[0]

# Absolute dataset root plus relative split folders lets Ultralytics find the same data from any working directory.
WORK.mkdir(exist_ok=True)
DATA_YAML = WORK / "insects.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": dict(enumerate(CLASS_NAMES)),
}, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

### Check one label visually

The following cell draws one training polygon on the original image, then shows it at the model input size. A valid shape can still be mislabeled. Check the outline, class, and apparent object size before launching training.

In [ ]:
from PIL import ImageDraw

image_path, positive_label = training_example
preview = Image.open(image_path).convert("RGB")
draw = ImageDraw.Draw(preview)
for row in positive_label.read_text(encoding="utf-8").splitlines():
    values = row.split()
    if values:
        class_id, coordinates = int(values[0]), list(map(float, values[1:]))
        # YOLO stores relative positions. Convert them back to pixel coordinates for PIL.
        pixels = [(x * (preview.width - 1), y * (preview.height - 1))
                  for x, y in zip(coordinates[::2], coordinates[1::2])]
        draw.polygon(pixels, outline="lime", width=2)
        print("Label:", CLASS_NAMES[class_id])
display(preview.copy().resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR))

## 3. Train the PyTorch model

For the first iteration, keep YOLOv8n-seg and 256 × 256 input. This reduces the changes needed in the firmware. Track box and mask mAP as well as false positives on background images. Use the validation split to tune training and keep the test split for evaluating the final models.

**Transfer learning** starts from the features in the pretrained YOLOv8n-seg checkpoint and adapts them to your examples. Its segmentation head must learn the class IDs in your dataset. An **epoch** is one pass over the training set. The default 50 epochs here is a starting point. Examine the training and validation curves to see whether the model is still improving or starting to overfit.

Use the training results folder for the recorded arguments and curves. `best.pt` is the checkpoint chosen by validation fitness. `last.pt` is the checkpoint from the final training epoch and can be used to resume a run. One class is enough to learn the workflow end to end.

To put training settings in a separate YAML, see the [training configuration map and example](../Doc/Source-Walkthrough.md#3-where-yolo-hyperparameters-live). The training cell prints the path to `args.yaml` for the actual run.

In [ ]:
from ultralytics import YOLO

EPOCHS = 50
DEVICE = None  # Use 0 for the first CUDA GPU or "cpu" to force CPU.
# Load pretrained segmentation weights so training can reuse existing visual features.
trainer = YOLO("yolov8n-seg.pt")
# Keep the image size consistent with the board. workers=0 avoids worker-spawn issues in Jupyter on Windows.
trainer.train(
    data=str(DATA_YAML), imgsz=IMAGE_SIZE, epochs=EPOCHS,
    project=str(WORK / "runs"), name="insect_yolov8n_seg",
    device=DEVICE, workers=0, seed=42,
)
# Read the actual path chosen by Ultralytics, including any run-name suffix.
BEST_PT = Path(trainer.trainer.best).resolve()
assert BEST_PT.is_file(), BEST_PT
print(BEST_PT)
print("Saved training arguments:", BEST_PT.parent.parent / "args.yaml")

## 4. Export a fully quantized TFLite model

In the dedicated Python environment, follow ST's per-channel export workflow. Open `ultralytics/engine/exporter.py` and change the `onnx2tf.convert` argument from `quant_type="per-tensor"` to `quant_type="per-channel"`. After editing, restart the kernel and rerun the configuration, reference, and dataset cells to restore the Python variables. If you already trained the model, skip retraining and set `BEST_PT` in the export cell to the checkpoint for that run.

The exporter uses the validation split of `insects.yaml` to calibrate with your own camera images. Keep the `*_full_integer_quant.tflite` file. The next cell checks its tensor types and shapes. Changes to the output order, layout, or meaning may require corresponding changes to the C postprocessor.

### What calibration changes

Weights store learned parameters, and activations hold intermediate values produced by a particular input. Full-integer quantization represents both with low-precision values. Calibration runs representative images through the graph to estimate activation ranges and choose scales and zero points. It does not update the learned weights. Include the backgrounds and lighting levels the board will actually see. For per-channel quantization, supported weight channels can each have their own scale. The I/O tensors inspected here expose the scalar parameters used at the model boundary.

Keep NMS outside this export. The STM32 implementation needs the raw candidate and prototype buffers so its C decoder can filter and reconstruct them.

In [ ]:
import inspect
from ultralytics import YOLO  # Keep this cell runnable after a kernel restart.
from ultralytics.engine import exporter

assert 'quant_type="per-channel"' in inspect.getsource(exporter), (
    f"Set per-channel quantization in {inspect.getfile(exporter)} and restart the kernel."
)

# After restarting the kernel, set BEST_PT to the checkpoint from your training run.
# BEST_PT = WORK / "runs" / "insect_yolov8n_seg" / "weights" / "best.pt"
fp_model = YOLO(str(BEST_PT))
assert list(fp_model.names.values()) == CLASS_NAMES
# Calibrate on val and leave NMS to the decoder. Batch 1 matches the camera snapshot flow.
exported = Path(fp_model.export(
    format="tflite", imgsz=IMAGE_SIZE, batch=1, int8=True, nms=False,
    data=str(DATA_YAML), split="val", device="cpu",
)).resolve()
# The exporter can write several TFLite variants. Select the file with integer input and output.
QUANTIZED = exported.parent / f"{BEST_PT.stem}_full_integer_quant.tflite"
assert QUANTIZED.is_file(), f"Look for the full_integer_quant file beside {exported}"

In [ ]:
candidate = open_tflite(QUANTIZED)
input_specs = [tensor_info(d) for d in candidate.get_input_details()]
output_specs = [tensor_info(d) for d in candidate.get_output_details()]
print(json.dumps({"input": input_specs, "outputs": output_specs}, indent=2))
assert len(input_specs) == 1 and input_specs[0]["shape"] == [1, 256, 256, 3]
assert input_specs[0]["dtype"] == "int8"
assert all(spec["dtype"] == "int8" and spec["scale"] > 0 for spec in output_specs)
assert len(output_specs) == 2
# At 256 × 256 the head uses grids 32² + 16² + 8² = 1344. Each row stores box, classes, and mask coefficients.
expected_shapes = {(1, 4 + len(CLASS_NAMES) + 32, 1344), (1, 64, 64, 32)}
assert {tuple(spec["shape"]) for spec in output_specs} == expected_shapes, (
    "Tensor shapes differ from the demo. Check the export and C integration."
)

# Pass the class order and original TFLite interface to Notebook 02. ST's generated interface is checked there.
manifest = {
    "model": str(QUANTIZED),
    "class_names": CLASS_NAMES,
    "imgsz": IMAGE_SIZE,
    "input": input_specs,
    "outputs": output_specs,
}
MANIFEST = WORK / "insect_model.json"
MANIFEST.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
print(QUANTIZED)
print(MANIFEST)

## 5. Evaluate both models on the test split

Compare box and mask mAP on the same split. Then inspect both models on the same resized image. Ultralytics evaluation uses its own preprocessing. For the final check, use tunnel images and the board's real preprocessing.

**Precision** is the fraction of reported objects that are correct. **Recall** is the fraction of labeled objects that are found. **IoU** measures overlap as intersection divided by union. `mAP50–95` averages average precision across IoU thresholds from 0.50 to 0.95 and across evaluated classes. Box mAP evaluates rectangles and mask mAP evaluates pixel regions. Check both before deciding whether quantization preserved the behavior you need.

For a trap, inspect empty scenes for false triggers and difficult images for missed insects. The visualization uses the demo's 0.5 confidence, 0.4 NMS IoU, and 10-object cap as a practical comparison. Those settings control individual predictions. The dataset-level validation call separately uses Ultralytics metric settings.

In [ ]:
models = {"PyTorch FP32": YOLO(str(BEST_PT)), "TFLite INT8": YOLO(str(QUANTIZED), task="segment")}
# Use the same test split and image size so data changes do not explain differences in the metrics.
for name, model in models.items():
    metrics = model.val(data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=1, device="cpu")
    print(name, {"box mAP50-95": metrics.box.map, "mask mAP50-95": metrics.seg.map})

sample = Image.open(first_test_image if IMAGE is None else IMAGE).convert("RGB")
sample = sample.resize((IMAGE_SIZE, IMAGE_SIZE), Image.Resampling.BILINEAR)
for name, model in models.items():
    result = model.predict(sample, imgsz=IMAGE_SIZE, conf=0.5, iou=0.4, max_det=10, device="cpu")[0]
    print(name)
    # Ultralytics plot() returns BGR. PIL expects RGB, so swap the red and blue channels.
    display(Image.fromarray(result.plot()[..., ::-1]))

Pass the `.tflite` and `insect_model.json` to [Notebook 02](02_Compile_and_Deploy_NUCLEO_N6.ipynb). If they move to another machine, set the new TFLite path in that notebook. Preserve the order of `class_names` from training.